<style>
div.mermaid > svg { width: 70% !important; height: auto !important; }
</style>

# 2D stencil: a heat-diffusion step

A **stencil** rewrites every grid point as a weighted sum of itself and its neighbors. The
classic example is a single **Jacobi heat-diffusion step** on a 2D grid, which is also just a
5-point local average. We write it two ways against the same reference: a straightforward
global-memory kernel, then a shared-memory-tiled version that stages each block's tile **plus a
one-cell halo** into SMEM so neighbor reads stay on-chip.

**You'll learn:** 2-D access over a flat `cutlass.Array` (`a[row*W + col]`); reading a point's
four neighbors with **zero-boundary** handling (out-of-grid neighbors count as 0); and the
**halo** pattern — staging a `(TS+2)x(TS+2)` shared-memory tile (the block's `TS x TS` output
region ringed by one row/column of neighbors) so the stencil reads from SMEM instead of
re-reading global memory up to five times per point.

**Runs on:** any CUDA GPU.

In [ ]:
import cutlass
import cutlass.cute as cute
import torch
import torch.nn.functional as F

# One Jacobi heat step == a 5-point average; out-of-grid neighbors count as 0:
#   out[i,j] = C0*in[i,j] + C1*(in[i-1,j] + in[i+1,j] + in[i,j-1] + in[i,j+1])
C0 = 0.2   # center weight   (baked at trace time -- plain Python floats)
C1 = 0.2   # neighbor weight
TS = 16    # block is TS x TS; H and W must be multiples of TS

## 1. The naive kernel

One thread owns one output point `(row, col)` and reads its four neighbors straight from global
memory. The boundary is handled by **skipping** any neighbor that falls off the grid — a skipped
neighbor contributes nothing, i.e. it counts as 0. Each `if` is a plain Python `if` on a *staged*
value, so the DSL turns it into a real GPU branch and never reads out of bounds.

In [ ]:
@cute.kernel
def stencil_naive_kernel(inp: cutlass.Array, out: cutlass.Array, H: cutlass.Int32, W: cutlass.Int32):
    tx, ty, _ = cute.arch.thread_idx()
    bx, by, _ = cute.arch.block_idx()
    bdx, bdy, _ = cute.arch.block_dim()
    row = bx * bdx + tx
    col = by * bdy + ty
    if row < H and col < W:
        acc = C0 * inp[row * W + col]
        # Add each in-grid neighbor; an out-of-grid neighbor is simply skipped (counts as 0).
        if row > 0:
            acc = acc + C1 * inp[(row - 1) * W + col]
        if row < H - 1:
            acc = acc + C1 * inp[(row + 1) * W + col]
        if col > 0:
            acc = acc + C1 * inp[row * W + (col - 1)]
        if col < W - 1:
            acc = acc + C1 * inp[row * W + (col + 1)]
        out[row * W + col] = acc

## 2. Shared-memory tiling with a halo

The naive kernel re-reads every point up to five times from global memory (each point is a
neighbor of four others). A block can instead stage its data **once** into shared memory and then
read neighbors on-chip. The catch: threads on the block's edge need neighbors that belong to the
*next* block — the **halo**. So the SMEM tile is `(TS+2) x (TS+2)`: the block's `TS x TS` output
region plus a one-cell ring around it.

Loading is cooperative — every thread writes its own center cell, and the edge threads
(`tx == 0`, `tx == TS-1`, `ty == 0`, `ty == TS-1`) additionally fetch the halo rows/columns,
using 0 where the halo falls off the grid. A `cute.arch.barrier()` then makes the full tile
visible before anyone reads it, and the stencil becomes four SMEM lookups.

In [ ]:
@cute.kernel
def stencil_smem_kernel(inp: cutlass.Array, out: cutlass.Array, H: cutlass.Int32, W: cutlass.Int32):
    tx, ty, _ = cute.arch.thread_idx()
    bx, by, _ = cute.arch.block_idx()
    # SMEM tile = the block's TS x TS region + a 1-cell halo on every side.
    tile = cutlass.Array(cutlass.Float32, (TS + 2, TS + 2), space=cutlass.AddressSpace.smem)
    row0 = bx * TS  # this block's top-left output cell in the grid
    col0 = by * TS
    row = row0 + tx
    col = col0 + ty

    # Center: every thread brings in its own cell (in-bounds since H, W are multiples of TS).
    tile[tx + 1, ty + 1] = inp[row * W + col]
    # Top / bottom halo rows: loaded by the first / last row of threads (0 if off the grid).
    if tx == 0:
        v = cutlass.Float32(0.0)
        if row0 - 1 >= 0:
            v = inp[(row0 - 1) * W + col]
        tile[0, ty + 1] = v
    if tx == TS - 1:
        v = cutlass.Float32(0.0)
        if row0 + TS < H:
            v = inp[(row0 + TS) * W + col]
        tile[TS + 1, ty + 1] = v
    # Left / right halo columns: loaded by the first / last column of threads.
    if ty == 0:
        v = cutlass.Float32(0.0)
        if col0 - 1 >= 0:
            v = inp[row * W + (col0 - 1)]
        tile[tx + 1, 0] = v
    if ty == TS - 1:
        v = cutlass.Float32(0.0)
        if col0 + TS < W:
            v = inp[row * W + (col0 + TS)]
        tile[tx + 1, TS + 1] = v

    # Don't read the tile until every thread has finished filling it.
    cute.arch.barrier()

    # Now the stencil is four on-chip lookups around the center cell.
    acc = C0 * tile[tx + 1, ty + 1] + C1 * (
        tile[tx, ty + 1] + tile[tx + 2, ty + 1] + tile[tx + 1, ty] + tile[tx + 1, ty + 2]
    )
    out[row * W + col] = acc

## 3. Run both and verify

A small factory bakes the chosen kernel into a launcher: one block per `TS x TS` output tile,
`block = (TS, TS, 1)`. We compare both kernels against a zero-padded NumPy-style reference; the
halo's zeros make the SMEM version agree with the naive one bit-for-bit (down to float epsilon).

In [ ]:
def make_stencil(kernel):
    @cute.jit
    def run(inp: cutlass.Array, out: cutlass.Array, H: cutlass.Int32, W: cutlass.Int32):
        grid = (H // TS, W // TS, 1)
        kernel(inp, out, H, W).launch(grid=grid, block=(TS, TS, 1))
    return run


def reference(x):
    """Host mirror: 5-point stencil with out-of-grid neighbors zeroed (zero-pad the grid)."""
    xp = F.pad(x, (1, 1, 1, 1))  # one-cell zero border -> (H+2, W+2)
    H, W = x.shape
    north, south = xp[0:H, 1:W + 1], xp[2:H + 2, 1:W + 1]
    west, east = xp[1:H + 1, 0:W], xp[1:H + 1, 2:W + 2]
    return C0 * x + C1 * (north + south + west + east)


H, W = 64, 64
assert H % TS == 0 and W % TS == 0, "H and W must be multiples of TS"
x = torch.randn(H, W, dtype=torch.float32, device="cuda")
ref = reference(x.cpu())

for name, kern in (("naive", stencil_naive_kernel), ("smem", stencil_smem_kernel)):
    out = torch.zeros(H, W, dtype=torch.float32, device="cuda")
    make_stencil(kern)(cute.runtime.from_dlpack(x), cute.runtime.from_dlpack(out), H, W)
    torch.testing.assert_close(out.cpu(), ref, atol=1e-4, rtol=1e-4)
    print(f"PASS  {name}")

# Expected output:
# PASS  naive
# PASS  smem

## Try it yourself

1. **Change the physics.** Set `C0 = 1.0` and `C1 = 0.25` for a sharper diffusion step, or
   `C0, C1 = 0.0, 0.25` for a pure neighbor-average (the center drops out). Update `reference`'s
   coefficients to match and re-run — both kernels still agree.
2. **Iterate the heat equation.** Wrap the launch in a Python loop, ping-ponging two buffers
   (`x -> out`, then `out -> x`), to watch a random field smooth out over N steps.
3. **Inspect the halo.** Temporarily set the halo loads to a sentinel (e.g. `-1.0`) instead of the
   real neighbor, and print a border row of `out`: the corrupted values show exactly which cells
   the halo feeds.
4. **Compare to `04_tiled_gemm`.** Both stage a tile into SMEM behind a `cute.arch.barrier()`; the
   stencil's twist is the one-cell halo, since each output also needs its neighbors' data.